In [1]:
import pandas as pd
df = pd.read_csv("employees_kenya.csv")
print(df.head())

  Employee ID               Full Name First Name Middle Name Last Name  \
0     EMP-001    Kevin Wairimu Njenga      Kevin     Wairimu    Njenga   
1     EMP-002  Wanjiku Wanjiku Mbatha    Wanjiku     Wanjiku    Mbatha   
2     EMP-003        Njeri Mugo Kuria      Njeri        Mugo     Kuria   
3     EMP-004    Peter Mohamed Mutuku      Peter     Mohamed    Mutuku   
4     EMP-005    Winnie Musyoka Kyalo     Winnie     Musyoka     Kyalo   

   Gender Date of Birth  National ID     KRA PIN     SHA Number  ...  \
0    Male    1981-12-21     13439167  A13439167B  SHA9963334018  ...   
1  Female    1998-08-18     15217026  A15217026C  SHA8574149614  ...   
2  Female    1986-06-07     17646997  A17646997H  SHA8615270538  ...   
3    Male    1981-11-16     22732411  A22732411L  SHA1240251661  ...   
4  Female    1995-03-26     29580115  A29580115E  SHA7010379415  ...   

   Bonus (KES)  Gross Salary (KES) NSSF Deduction (KES) PAYE (KES)  \
0         1000               43000                 2

In [2]:
df.dtypes

Employee ID             object
Full Name               object
First Name              object
Middle Name             object
Last Name               object
Gender                  object
Date of Birth           object
National ID              int64
KRA PIN                 object
SHA Number              object
NSSF Number              int64
Phone Number             int64
Email                   object
Department              object
Position                object
Seniority Level         object
Hire Date               object
Basic Salary (KES)       int64
Bonus (KES)              int64
Gross Salary (KES)       int64
NSSF Deduction (KES)     int64
PAYE (KES)               int64
Net Salary (KES)         int64
Bank Name               object
Bank Account No          int64
Leave Days Remaining     int64
Employment Status       object
Title                   object
dtype: object

In [3]:
df["National ID"]      = df["National ID"].astype(str)
df["NSSF Number"]      = df["NSSF Number"].astype(str)
df["Phone Number"]     = df["Phone Number"].astype(str).apply(lambda x: "0" + x if not x.startswith("0") else x)
df["Bank Account No"]  = df["Bank Account No"].astype(str)

# Fix dates from string to actual datetime
df["Date of Birth"]    = pd.to_datetime(df["Date of Birth"])
df["Hire Date"]        = pd.to_datetime(df["Hire Date"])

# Verify
print(df.dtypes)

Employee ID                     object
Full Name                       object
First Name                      object
Middle Name                     object
Last Name                       object
Gender                          object
Date of Birth           datetime64[ns]
National ID                     object
KRA PIN                         object
SHA Number                      object
NSSF Number                     object
Phone Number                    object
Email                           object
Department                      object
Position                        object
Seniority Level                 object
Hire Date               datetime64[ns]
Basic Salary (KES)               int64
Bonus (KES)                      int64
Gross Salary (KES)               int64
NSSF Deduction (KES)             int64
PAYE (KES)                       int64
Net Salary (KES)                 int64
Bank Name                       object
Bank Account No                 object
Leave Days Remaining     

In [4]:
df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(" ", "_")
)
print(df.columns)

Index(['employee_id', 'full_name', 'first_name', 'middle_name', 'last_name',
       'gender', 'date_of_birth', 'national_id', 'kra_pin', 'sha_number',
       'nssf_number', 'phone_number', 'email', 'department', 'position',
       'seniority_level', 'hire_date', 'basic_salary_(kes)', 'bonus_(kes)',
       'gross_salary_(kes)', 'nssf_deduction_(kes)', 'paye_(kes)',
       'net_salary_(kes)', 'bank_name', 'bank_account_no',
       'leave_days_remaining', 'employment_status', 'title'],
      dtype='object')


In [5]:
# Remove parentheses and _kes suffix for cleaner column names
df.columns = df.columns.str.replace(r'_\(kes\)', '', regex=True)
print(df.columns.tolist())

['employee_id', 'full_name', 'first_name', 'middle_name', 'last_name', 'gender', 'date_of_birth', 'national_id', 'kra_pin', 'sha_number', 'nssf_number', 'phone_number', 'email', 'department', 'position', 'seniority_level', 'hire_date', 'basic_salary', 'bonus', 'gross_salary', 'nssf_deduction', 'paye', 'net_salary', 'bank_name', 'bank_account_no', 'leave_days_remaining', 'employment_status', 'title']


In [6]:
# checking for duplicates
print(df.duplicated().sum())


0


In [7]:
# check for missing values
print(df.isnull().sum())

employee_id             0
full_name               0
first_name              0
middle_name             0
last_name               0
gender                  0
date_of_birth           0
national_id             0
kra_pin                 0
sha_number              0
nssf_number             0
phone_number            0
email                   0
department              0
position                0
seniority_level         0
hire_date               0
basic_salary            0
bonus                   0
gross_salary            0
nssf_deduction          0
paye                    0
net_salary              0
bank_name               0
bank_account_no         0
leave_days_remaining    0
employment_status       0
title                   0
dtype: int64


In [8]:
from sqlalchemy import create_engine, text
from sqlalchemy.exc import SQLAlchemyError

In [9]:
import os
print(os.getcwd())

c:\Users\Cameline\Desktop\Employee Management System


In [10]:
from dotenv import load_dotenv
from urllib.parse import quote_plus
from sqlalchemy import create_engine, text
import psycopg2
import os

# Force load from explicit path
load_dotenv(r"C:\Users\Cameline\Desktop\Employee Management System\.env", override=True)


# psycopg2 connection
conn = psycopg2.connect(
    host=os.getenv("DB_HOST"),
    database=os.getenv("DB_NAME"),
    user=os.getenv("DB_USERNAME"),
    password=os.getenv("DB_PASSWORD"),
    port=os.getenv("DB_PORT")
)

# SQLAlchemy engine
password = quote_plus(os.getenv("DB_PASSWORD"))
engine = create_engine(
    f"postgresql+psycopg2://{os.getenv('DB_USERNAME')}:{password}@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)

print(" Connected successfully")

 Connected successfully


In [11]:
with engine.connect() as conn:
    result = conn.execute(text("SELECT table_name FROM information_schema.tables WHERE table_schema = 'public';"))
    print(result.fetchall())

[('employees',), ('positions',), ('attendance',), ('payroll',)]


In [12]:
employees_df = df[[
    'employee_id', 'title', 'first_name', 'middle_name', 'last_name',
    'full_name', 'gender', 'date_of_birth', 'national_id', 'kra_pin',
    'sha_number', 'nssf_number', 'phone_number', 'email',
    'bank_name', 'bank_account_no', 'employment_status', 'hire_date',
    'leave_days_remaining'
]]

positions_df = df[[
    'employee_id', 'department', 'position', 'seniority_level',
    'basic_salary', 'bonus', 'gross_salary',
    'nssf_deduction', 'paye', 'net_salary'
]]

print("Employees shape:", employees_df.shape)
print("Positions shape:", positions_df.shape)

Employees shape: (300, 19)
Positions shape: (300, 10)


In [13]:
import sqlalchemy

with engine.connect() as conn:
    conn.execute(text("DROP TABLE IF EXISTS positions CASCADE;"))
    conn.execute(text("DROP TABLE IF EXISTS employees CASCADE;"))
    conn.commit()

# Insert employees table
employees_df.to_sql(
    'employees',
    engine,
    if_exists='replace',
    index=False,
    dtype={
        'employee_id':        sqlalchemy.types.VARCHAR(20),
        'title':              sqlalchemy.types.VARCHAR(10),
        'first_name':         sqlalchemy.types.VARCHAR(50),
        'middle_name':        sqlalchemy.types.VARCHAR(50),
        'last_name':          sqlalchemy.types.VARCHAR(50),
        'full_name':          sqlalchemy.types.VARCHAR(100),
        'gender':             sqlalchemy.types.VARCHAR(10),
        'date_of_birth':      sqlalchemy.types.Date(),
        'national_id':        sqlalchemy.types.VARCHAR(20),
        'kra_pin':            sqlalchemy.types.VARCHAR(20),
        'sha_number':         sqlalchemy.types.VARCHAR(20),
        'nssf_number':        sqlalchemy.types.VARCHAR(20),
        'phone_number':       sqlalchemy.types.VARCHAR(15),
        'email':              sqlalchemy.types.VARCHAR(100),
        'bank_name':          sqlalchemy.types.VARCHAR(50),
        'bank_account_no':    sqlalchemy.types.VARCHAR(20),
        'employment_status':  sqlalchemy.types.VARCHAR(20),
        'hire_date':          sqlalchemy.types.Date(),
        'leave_days_remaining': sqlalchemy.types.Integer()
    }
)

# Add primary key
with engine.connect() as conn:
    conn.execute(text("ALTER TABLE employees ADD PRIMARY KEY (employee_id);"))
    conn.commit()

print(" employees table inserted with primary key")

 employees table inserted with primary key


In [14]:

positions_df.to_sql(
    'positions',
    engine,
    if_exists='replace',
    index=False,
    dtype={
        'employee_id':     sqlalchemy.types.VARCHAR(20),
        'department':      sqlalchemy.types.VARCHAR(50),
        'position':        sqlalchemy.types.VARCHAR(50),
        'seniority_level': sqlalchemy.types.VARCHAR(20),
        'basic_salary':    sqlalchemy.types.Integer(),
        'bonus':           sqlalchemy.types.Integer(),
        'gross_salary':    sqlalchemy.types.Integer(),
        'nssf_deduction':  sqlalchemy.types.Integer(),
        'paye':            sqlalchemy.types.Integer(),
        'net_salary':      sqlalchemy.types.Integer()
    }
)

# Add primary key + foreign key
with engine.connect() as conn:
    conn.execute(text("ALTER TABLE positions ADD PRIMARY KEY (employee_id);"))
    conn.execute(text("""
        ALTER TABLE positions 
        ADD CONSTRAINT fk_positions_employee 
        FOREIGN KEY (employee_id) REFERENCES employees(employee_id);
    """))
    conn.commit()

print("✅ positions table inserted with FK to employees")

✅ positions table inserted with FK to employees


In [15]:
with engine.connect() as conn:
    tables = conn.execute(text(
        "SELECT table_name FROM information_schema.tables WHERE table_schema = 'public';"
    ))
    print("Tables:", tables.fetchall())
    
    emp_count = conn.execute(text("SELECT COUNT(*) FROM employees;"))
    pos_count = conn.execute(text("SELECT COUNT(*) FROM positions;"))
    print("Employee rows:", emp_count.fetchone()[0])
    print("Position rows:", pos_count.fetchone()[0])

Tables: [('employees',), ('positions',), ('attendance',), ('payroll',)]
Employee rows: 300
Position rows: 300


In [16]:

import random
from datetime import date

# Get all employee IDs and their starting leave days
employee_ids = employees_df['employee_id'].tolist()
leave_balance = dict(zip(employees_df['employee_id'], employees_df['leave_days_remaining']))

# Get all working days in 2024 (Monday to Friday only)
all_working_days = pd.bdate_range(start='2024-01-01', end='2024-12-31').tolist()

rows = []
attendance_id = 1

for emp_id in employee_ids:
    remaining_leave = leave_balance[emp_id]  # start with their actual leave balance
    
    for day in all_working_days:
        # Assign status randomly but realistically
        rand = random.random()
        if rand < 0.85:
            status = 'Present'
        elif rand < 0.93:
            status = 'Absent'
        else:
            # Only use Leave if they still have days remaining
            if remaining_leave > 0:
                status = 'Leave'
                remaining_leave -= 1
            else:
                status = 'Absent'

        rows.append({
            'attendance_id':        attendance_id,
            'employee_id':          emp_id,
            'date':                 day.date(),
            'status':               status,
            'leave_days_remaining': remaining_leave
        })
        attendance_id += 1

attendance_df = pd.DataFrame(rows)
print(f" Attendance records generated: {len(attendance_df):,} rows")
print(attendance_df.head(10))

 Attendance records generated: 78,600 rows
   attendance_id employee_id        date   status  leave_days_remaining
0              1     EMP-001  2024-01-01  Present                    46
1              2     EMP-001  2024-01-02  Present                    46
2              3     EMP-001  2024-01-03   Absent                    46
3              4     EMP-001  2024-01-04  Present                    46
4              5     EMP-001  2024-01-05   Absent                    46
5              6     EMP-001  2024-01-08  Present                    46
6              7     EMP-001  2024-01-09   Absent                    46
7              8     EMP-001  2024-01-10  Present                    46
8              9     EMP-001  2024-01-11  Present                    46
9             10     EMP-001  2024-01-12    Leave                    45


In [17]:


with engine.connect() as conn:
    conn.execute(text("DROP TABLE IF EXISTS attendance CASCADE;"))
    conn.commit()


attendance_df.to_sql(
    'attendance',
    engine,
    if_exists='replace',
    index=False,
    dtype={
        'attendance_id':        sqlalchemy.types.Integer(),
        'employee_id':          sqlalchemy.types.VARCHAR(20),
        'date':                 sqlalchemy.types.Date(),
        'status':               sqlalchemy.types.VARCHAR(20),
        'leave_days_remaining': sqlalchemy.types.Integer()
    }
)

# Add primary key and foreign key
with engine.connect() as conn:
    conn.execute(text("ALTER TABLE attendance ADD PRIMARY KEY (attendance_id);"))
    conn.execute(text("""
        ALTER TABLE attendance
        ADD CONSTRAINT fk_attendance_employee
        FOREIGN KEY (employee_id) REFERENCES employees(employee_id);
    """))
    conn.commit()

print(" Attendance table inserted with PK and FK!")
with engine.connect() as conn:
    count = conn.execute(text("SELECT COUNT(*) FROM attendance;"))
    print("Total attendance rows:", count.fetchone()[0])
    
    sample = conn.execute(text("SELECT * FROM attendance LIMIT 5;"))
    for row in sample:
        print(row)

 Attendance table inserted with PK and FK!
Total attendance rows: 78600
(1, 'EMP-001', datetime.date(2024, 1, 1), 'Present', 46)
(2, 'EMP-001', datetime.date(2024, 1, 2), 'Present', 46)
(3, 'EMP-001', datetime.date(2024, 1, 3), 'Absent', 46)
(4, 'EMP-001', datetime.date(2024, 1, 4), 'Present', 46)
(5, 'EMP-001', datetime.date(2024, 1, 5), 'Absent', 46)


In [18]:


months = range(1, 13)  # January to December
year = 2024

rows = []
payroll_id = 1

for _, emp in positions_df.iterrows():
    for month in months:
        # Last 2 months have some Pending payments — rest are Paid
        if month >= 11:
            payment_status = random.choice(['Paid', 'Pending'])
        else:
            payment_status = 'Paid'

        rows.append({
            'payroll_id':      payroll_id,
            'employee_id':     emp['employee_id'],
            'month':           month,
            'year':            year,
            'basic_salary':    emp['basic_salary'],
            'bonus':           emp['bonus'],
            'gross_salary':    emp['gross_salary'],
            'nssf_deduction':  emp['nssf_deduction'],
            'paye':            emp['paye'],
            'net_salary':      emp['net_salary'],
            'payment_status':  payment_status
        })
        payroll_id += 1

payroll_df = pd.DataFrame(rows)
print(f" Payroll records generated: {len(payroll_df):,} rows")
print(payroll_df.head(10))

 Payroll records generated: 3,600 rows
   payroll_id employee_id  month  year  basic_salary  bonus  gross_salary  \
0           1     EMP-001      1  2024         42000   1000         43000   
1           2     EMP-001      2  2024         42000   1000         43000   
2           3     EMP-001      3  2024         42000   1000         43000   
3           4     EMP-001      4  2024         42000   1000         43000   
4           5     EMP-001      5  2024         42000   1000         43000   
5           6     EMP-001      6  2024         42000   1000         43000   
6           7     EMP-001      7  2024         42000   1000         43000   
7           8     EMP-001      8  2024         42000   1000         43000   
8           9     EMP-001      9  2024         42000   1000         43000   
9          10     EMP-001     10  2024         42000   1000         43000   

   nssf_deduction  paye  net_salary payment_status  
0            2160  4635       36205           Paid  
1      

In [19]:
import sqlalchemy

with engine.connect() as conn:
    conn.execute(text("DROP TABLE IF EXISTS payroll CASCADE;"))
    conn.commit()


payroll_df.to_sql(
    'payroll',
    engine,
    if_exists='replace',
    index=False,
    dtype={
        'payroll_id':     sqlalchemy.types.Integer(),
        'employee_id':    sqlalchemy.types.VARCHAR(20),
        'month':          sqlalchemy.types.Integer(),
        'year':           sqlalchemy.types.Integer(),
        'basic_salary':   sqlalchemy.types.Integer(),
        'bonus':          sqlalchemy.types.Integer(),
        'gross_salary':   sqlalchemy.types.Integer(),
        'nssf_deduction': sqlalchemy.types.Integer(),
        'paye':           sqlalchemy.types.Integer(),
        'net_salary':     sqlalchemy.types.Integer(),
        'payment_status': sqlalchemy.types.VARCHAR(20)
    }
)

# Add primary key and foreign key
with engine.connect() as conn:
    conn.execute(text("ALTER TABLE payroll ADD PRIMARY KEY (payroll_id);"))
    conn.execute(text("""
        ALTER TABLE payroll
        ADD CONSTRAINT fk_payroll_employee
        FOREIGN KEY (employee_id) REFERENCES employees(employee_id);
    """))
    conn.commit()

print(" Payroll table inserted with PK and FK!")

 Payroll table inserted with PK and FK!


In [20]:
with engine.connect() as conn:
    # Check all tables exist
    tables = conn.execute(text(
        "SELECT table_name FROM information_schema.tables WHERE table_schema = 'public';"
    ))
    print(" Tables in database:", [row[0] for row in tables])

    # Check row counts
    for table in ['employees', 'positions', 'attendance', 'payroll']:
        count = conn.execute(text(f"SELECT COUNT(*) FROM {table};"))
        print(f"  {table}: {count.fetchone()[0]:,} rows")

    # Quick payroll sample
    sample = conn.execute(text("SELECT * FROM payroll LIMIT 5;"))
    print("\n Sample payroll rows:")
    for row in sample:
        print(row)

 Tables in database: ['employees', 'positions', 'attendance', 'payroll']
  employees: 300 rows
  positions: 300 rows
  attendance: 78,600 rows
  payroll: 3,600 rows

 Sample payroll rows:
(1, 'EMP-001', 1, 2024, 42000, 1000, 43000, 2160, 4635, 36205, 'Paid')
(2, 'EMP-001', 2, 2024, 42000, 1000, 43000, 2160, 4635, 36205, 'Paid')
(3, 'EMP-001', 3, 2024, 42000, 1000, 43000, 2160, 4635, 36205, 'Paid')
(4, 'EMP-001', 4, 2024, 42000, 1000, 43000, 2160, 4635, 36205, 'Paid')
(5, 'EMP-001', 5, 2024, 42000, 1000, 43000, 2160, 4635, 36205, 'Paid')
